# Experimental local self-play training

This notebook **does update model weights** after completed games. The simulator is incomplete: it uses implemented cards, not the full Standard pool, and cannot establish the best Standard deck or strategy. Check notebook 09 for current coverage. No Mercenaries or Battlegrounds.

Choose your budget, then run the last cell. The policy plays both seats, and learns from each player's own result. Games that reach the action limit are saved as capped and do not update weights. Progress shows games completed; each completed episode saves a new model snapshot. This is a small linear baseline, not a strong-player claim.

Interrupting stops the child process. Previously saved snapshots remain; the interrupted game is not checkpointed. To resume, set RESUME_RECEIPT to a saved episode receipt path, keep seed, learning rate and action limit unchanged, and set EPISODES to the number of additional games. The runner restores the policy and continues the match schedule. Engine or runner changes invalidate exact resume. Each run has a new folder and never replaces older models.


In [ ]:
from pathlib import Path
import subprocess, sys, json
from IPython.display import display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_candidate.py').exists()), None)
if ROOT is None: raise RuntimeError('Open from the death-knight-lab folder.')
EPISODES = 1
MAX_ACTIONS = 500
SEED = 71
LEARNING_RATE = 0.001
RESUME_RECEIPT = None  # Or an absolute path to episode-00001.json from a previous run
print(f'Experimental budget: {EPISODES} games; at most {EPISODES * MAX_ACTIONS} actions. Learning enabled.')


In [ ]:
command = [sys.executable, str(ROOT/'tools/train_candidate.py'), '--engine-root', str(ROOT/'staging/rebased-88'), '--episodes', str(EPISODES), '--max-actions', str(MAX_ACTIONS), '--seed', str(SEED), '--learning-rate', str(LEARNING_RATE), '--output', str(ROOT/'runs/candidate_training'), '--experimental']
if RESUME_RECEIPT is not None: command += ['--resume-receipt', str(RESUME_RECEIPT)]
progress = display('Training: 0 completed games', display_id=True)
completed = False
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            try: item = json.loads(line)
            except ValueError:
                print(line.rstrip()); continue
            if item['event']=='started': print('Run folder:', item['folder'])
            elif item['event']=='progress': print(f"Playing game {item['episode']}/{item['total']}")
            elif item['event']=='saved':
                done, total = item['episode'], item['total']
                filled = int(20 * done / total)
                progress.update(f"[{'#' * filled}{'-' * (20-filled)}] {done}/{total} games")
                result = item['result']
                print(f"Game {done}: {result['steps']} actions; capped={result['truncated']}; update={result['updated']}")
                print('Saved model:', item['checkpoint']['path'])
                print('Resume receipt:', item['receipt'])
            elif item['event']=='complete':
                completed = True; print('Completed run manifest:', item['manifest'])
        code = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=5)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if code or not completed: raise RuntimeError('Run stopped early. Previously saved model snapshots remain available.')
